# 1 · Open weather data, and a first forecast

Download hourly wind for any place, score two forecasts that need no learning, and find the best straight-line forecast with a formula. Background and derivations are in the [notes](notes/01_open_data.md).

Weather data by [Open-Meteo.com](https://open-meteo.com/), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); it is ERA5 from Copernicus, and needs no account.

The tools: numpy, the download and the score from `windlab.py`, and the figures from `plots.py`.

In [ ]:
import numpy as np
from windlab import fetch_wind, rmse, year
from plots import hardest_week, plot_best_lines, plot_first_look, plot_forecasts, plot_rmse_by_horizon

## 1 · One request

The API returns the wind speed and the direction $\varphi$ it blows **from**. We turn them into the two components u and v, which the networks of notebook 2 use:

$$u = -\text{speed}\cdot\sin\varphi \;\;(\text{towards east}), \qquad v = -\text{speed}\cdot\cos\varphi \;\;(\text{towards north}).$$

One request downloads two years of hourly wind at one point and converts it to u and v. The forecasts in this notebook are for the wind **speed**, $\sqrt{u^2 + v^2}$, the number a weather app shows.

In [ ]:
LAT, LON = 48.15, 17.11  # Bratislava: change to your town
time, u, v = fetch_wind(LAT, LON, "2023-01-01", "2024-12-31")
u, v = u[:, 0], v[:, 0]  # a single point
speed = np.hypot(u, v)  # the wind speed, in m/s
print(f"{len(time)} hours, {time[0]} .. {time[-1]}, mean speed {speed.mean():.2f} m/s")

## 2 · A first look

Left, the speed (black), u and v over the first two weeks; right, the average speed for every hour of the day.

In [ ]:
plot_first_look(time, u, v)

**Look for:** the wind changes over hours to days, rarely from one hour to the next, and it is strongest in the early afternoon (13 UTC, 14–15 h local time). The dip at 10 UTC is not weather: it is a seam in the ERA5 data (see the notes).

## 3 · Two forecasts that need no learning

| Symbol | Meaning, the same in all notebooks |
| --- | --- |
| $x$ | the **input**: the wind now, at hour $t$ |
| $y$ | the **target**: the true wind $h$ hours later |
| $\hat y$ | a **forecast** of $y$; its error is $\hat y - y$ |
| $\bar y$ | the average of $y$ over the training hours |

- Here the wind is its **speed**: $x$ is the speed now, $y$ the speed $h$ hours later.
- **Persistence** $\hat y = x$: "it stays as it is now". **Climatology** $\hat y = \bar y$: the average speed.
- **RMSE** $= \sqrt{\tfrac1N\sum(\hat y - y)^2}$, the typical miss in m/s. **Skill** $= 1 - \mathrm{RMSE}^2_\text{model}/\mathrm{RMSE}^2_\text{persistence}$, the standard skill score with persistence as the reference: 0.1 means a 10 % smaller mean squared error than persistence.
- We fit on 2023 and score on 2024: a forecast has to work on a future it has not seen.

The code scores both forecasts on 2024, for every horizon from 1 to 48 hours; the plot shows their RMSE against the horizon.

In [ ]:
train_hours, test_hours = np.flatnonzero(year(time) == 2023), np.flatnonzero(year(time) == 2024)
horizons = np.arange(1, 49)
rmse_persistence = np.array([rmse(speed[test_hours[:-h]], speed[test_hours[:-h] + h]) for h in horizons])
rmse_climatology = rmse(speed[train_hours].mean(), speed[test_hours])
plot_rmse_by_horizon(horizons, {"persistence": rmse_persistence}, rmse_climatology)
print(f"persistence: {rmse_persistence[0]:.2f} m/s at 1 h, {rmse_persistence[5]:.2f} m/s at 6 h; climatology: {rmse_climatology:.2f} m/s")

**Look for:** persistence is excellent one hour ahead and gets worse hour by hour, up to about 40 hours, where the daily cycle makes it dip slightly; beyond about 10 hours, climatology wins. That is where the correlation between the speed now and $h$ hours later, its **autocorrelation**, drops below ½ (why, and how the daily cycle enters: see the notes).

## 4 · The best straight line

**The model** is a line $\hat y = w\,x + b$ with two **parameters**, the **weight** $w$ and the **bias** $b$. Both baselines are such lines: persistence is $(w, b) = (1, 0)$, climatology $(0, \bar y)$.

**The problem:** of all lines, find the one with the smallest mean squared error on the training hours, the **loss**

$$L(w, b) = \frac1N\sum\left(w\,x + b - y\right)^2, \qquad (w^*, b^*) = \text{the } (w, b) \text{ at which } L \text{ is smallest}.$$

**The solution, least squares.** At the lowest point both slopes of $L$ are zero, $\partial L/\partial w = 0$ and $\partial L/\partial b = 0$: two linear equations in $w$ and $b$, whose solution is

$$w^* = \frac{\overline{(x-\bar x)(y-\bar y)}}{\overline{(x-\bar x)^2}} = \frac{\text{covariance of } x \text{ and } y}{\text{variance of } x}, \qquad b^* = \bar y - w^*\,\bar x,$$

where a bar is the average over the training hours (the derivation, and why this is the lowest point, are in the notes).

**What it means.** With this $b^*$ the forecast is $\hat y = \bar y + w^*\,(x - \bar x)$: the average, plus the share $w^*$ of the present deviation from the average. That share is the wind's **memory**: $w^* = 1$ keeps all of the deviation (persistence), $w^* = 0$ forgets it (climatology). And as both baselines are lines, the best line cannot do worse than either of them on the training hours.

The code writes the formula as a function and fits the speed in 2023 for 1, 6 and 24 hours ahead. Each panel shows the data (grey), persistence (dashed), climatology (orange) and the best line (green).

In [ ]:
def best_line(x, y):
    # Least squares: the w and b with the smallest mean squared error of w * x + b against y
    w = np.mean((x - x.mean()) * (y - y.mean())) / np.mean((x - x.mean()) ** 2)
    b = y.mean() - w * x.mean()
    return w, b

plot_best_lines({h: (speed[train_hours[:-h]], speed[train_hours[:-h] + h]) for h in (1, 6, 24)}, best_line)  # x = now, y = in h hours

**Look for:** the further ahead, the flatter the line, sliding from persistence ($w = 0.96$ at 1 hour) towards climatology ($w = 0.23$ at 24 hours).

Now the line is fitted for every horizon and scored on 2024; the plot adds it to the RMSE curves of section 3.

In [ ]:
rmse_line = []
for h in horizons:
    w, b = best_line(speed[train_hours[:-h]], speed[train_hours[:-h] + h])  # fitted on 2023
    rmse_line.append(rmse(w * speed[test_hours[:-h]] + b, speed[test_hours[:-h] + h]))  # scored on 2024
plot_rmse_by_horizon(horizons, {"persistence": rmse_persistence, "best line": rmse_line}, rmse_climatology)

Errors are averages; one week shows what they hide. The code fits the line 6 hours ahead and picks the week of 2024 in which persistence misses most; the plot shows the truth (black) and the three forecasts.

In [ ]:
h = 6
x, y = speed[train_hours[:-h]], speed[train_hours[:-h] + h]  # 2023, to fit
x_test, y_test = speed[test_hours[:-h]], speed[test_hours[:-h] + h]  # 2024, to score
w, b = best_line(x, y)
week = hardest_week(x_test - y_test)  # the week in which persistence misses most
plot_forecasts(time[test_hours[week] + h], y_test[week], {
    "persistence": x_test[week],
    "climatology": np.full(len(week), y.mean()),
    "best line": w * x_test[week] + b,
}, ylabel=f"y: speed in {h} h (m/s)", title="2024: the week in which persistence does worst")

**Look for:** the line beats both baselines at every horizon. In the hard week, though, it is the persistence curve shrunk towards the average: it turns 6 hours after the truth, exactly like persistence, because its only input is the wind now. Even the daily rise and fall, which the clock alone could foresee, comes 6 hours late.

## Your turn

1. Set `LAT, LON` to your home town and run the notebook again. Is it windier? From which horizon does climatology win?
2. Add `speed = u` at the start of section 3, to forecast the eastward component instead. Does it keep its memory longer than the speed?

**Next:** notebook 2 finds such a line without the formula, by walking downhill.